# Dictionary Generation

In [ ]:
# package imports

import compgenlnc as cgl

In [ ]:
# setting library
# change settings according to your environment

# base_dir = cgl.config.PACKAGE_DIR
base_dir = "."
cgl.set_base_path(base_dir)

conda_env = cgl.config.conda_environment
cgl.set_conda_env(conda_env)

doc2vec_seed = cgl.config.doc2vec_seed
cgl.set_doc2vec_seed(doc2vec_seed)

log_file = cgl.config.log_file
cgl.set_log_file(log_file)

In [ ]:
# prepare base files
basefiles_dir = None

cgl.fetch_lncrna_files()
cgl.fetch_mirna_files()
cgl.fetch_interaction_files()

filter_by_index = True
keep_data = False
maxsize = 500
cgl.gen_source_data(filter_by_index, keep_data, maxsize)

In [ ]:
# generate dictionaries for RNA sequences

records = cgl.load_seq_ss(cgl.paths.LNCRNA_SEQ_SS_DIR)

def get_dict_path(path):
    return cgl.paths.LNCRNA_DICTS_DIR / path

overwrite = True
cgl.gen_kmer_dict(get_dict_path(cgl.paths.KMER_DICT), records, 4, overwrite=overwrite)
cgl.gen_ctd_dict(get_dict_path(cgl.paths.CTD_DICT), records, overwrite=overwrite)
cgl.gen_doc2vec_dict(get_dict_path(cgl.paths.DOC2VEC_DICT), records)

kmer = cgl.load_dict(get_dict_path(cgl.paths.KMER_DICT))
ctd = cgl.load_dict(get_dict_path(cgl.paths.CTD_DICT))
d2v = cgl.load_dict(get_dict_path(cgl.paths.DOC2VEC_DICT))

cgl.gen_role2vec_dict(get_dict_path(cgl.paths.ROLE2VEC_DICT), kmer, ctd, d2v)
cgl.gen_normalized_sequence_dict(get_dict_path(cgl.paths.NORMAL_SEQ_DICT), records)
cgl.gen_loops_dict(get_dict_path(cgl.paths.LOOPS_DICT), cgl.paths.LNCRNA_LOOPS_DIR, records)

In [ ]:
# index dictionaries

def get_dict_path(path):
    return cgl.paths.LNCRNA_DICTS_DIR / path

def npy_path(path):
    return cgl.paths.DATASET_DIR / f"{cgl.LNCRNA}_{path}"

kmer = cgl.load_dict(get_dict_path(cgl.paths.KMER_DICT))
ctd = cgl.load_dict(get_dict_path(cgl.paths.CTD_DICT))
d2v = cgl.load_dict(get_dict_path(cgl.paths.DOC2VEC_DICT))
r2v = cgl.load_dict(get_dict_path(cgl.paths.ROLE2VEC_DICT))
normal = cgl.load_dict(get_dict_path(cgl.paths.NORMAL_SEQ_DICT))
loops = cgl.load_dict(get_dict_path(cgl.paths.LOOPS_DICT))

cgl.dict_to_numpy(npy_path(cgl.paths.KMER_DICT), kmer, rna=cgl.LNCRNA)
cgl.dict_to_numpy(npy_path(cgl.paths.CTD_DICT), ctd, rna=cgl.LNCRNA)
cgl.dict_to_numpy(npy_path(cgl.paths.DOC2VEC_DICT), d2v, rna=cgl.LNCRNA)
cgl.dict_to_numpy(npy_path(cgl.paths.ROLE2VEC_DICT), r2v, rna=cgl.LNCRNA)
cgl.dict_to_numpy(npy_path(cgl.paths.NORMAL_SEQ_DICT), normal, rna=cgl.LNCRNA)
cgl.dict_to_numpy(npy_path(cgl.paths.LOOPS_DICT), loops, rna=cgl.LNCRNA)

In [ ]:
# generate dictionaries for interaction subsequences

records = cgl.index_lncrna()
interactions = cgl.interactions_from_index()
binding_zones = list(cgl.read_binding_zones_from_list(cgl.paths.BINDING_DIR, interactions))
subrecords = list(cgl.get_subrecord_list(records, interactions, binding_zones, rna=cgl.LNCRNA))

def get_dict_path(path):
    return cgl.paths.LNCRNA_DICTS_DIR / f"{cgl.paths.BINDING_PREFIX}{path}"

overwrite = True
cgl.gen_kmer_dict(get_dict_path(cgl.paths.KMER_DICT), subrecords, 4, overwrite=overwrite)
cgl.gen_ctd_dict(get_dict_path(cgl.paths.CTD_DICT), subrecords, overwrite=overwrite)
cgl.gen_doc2vec_dict(get_dict_path(cgl.paths.DOC2VEC_DICT), subrecords)

kmer = cgl.load_dict(get_dict_path(cgl.paths.KMER_DICT))
ctd = cgl.load_dict(get_dict_path(cgl.paths.CTD_DICT))
d2v = cgl.load_dict(get_dict_path(cgl.paths.DOC2VEC_DICT))

cgl.gen_role2vec_dict(get_dict_path(cgl.paths.ROLE2VEC_DICT), kmer, ctd, d2v)
cgl.gen_normalized_sequence_dict(get_dict_path(cgl.paths.NORMAL_SEQ_DICT), subrecords)

loops_counters = [cgl.read_2d_structure_loops(cgl.paths.LNCRNA_LOOPS_DIR, rec.id) for rec in records]
cgl.gen_substructure_loops_dict(get_dict_path(cgl.paths.LOOPS_DICT), interactions, loops_counters, binding_zones, rna=cgl.LNCRNA)